In [1]:
import kagglehub
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
#pd.set_option('display.max_rows', None)

dataset_owner = "quackphuc/egoblind-short-context-frames"
path_train = kagglehub.dataset_download(
    dataset_owner,
    "train.csv"
)

path_val = kagglehub.dataset_download(
    dataset_owner,
    "val.csv"
)

path_test = kagglehub.dataset_download(
    dataset_owner,
    "test.csv"
)


In [2]:

#the csv file that corresponds to the video snippets

df_train = pd.read_csv(path_train)
df_val = pd.read_csv(path_val)
df_test = pd.read_csv(path_test)

In [3]:
df_train.describe()
df_train

,question_id,question,answer0,answer1,answer2,answer3,type,frame_folder_path
0,v_00000_1,Is the driver looking at me or facing forward?,The driver is facing forward.,Facing forward.,He is facing forward.,NaN,communication and interaction,train/v_00000_1
1,v_00000_2,Is everything secure inside the car?,Yes.,NaN,NaN,NaN,safety warnings,train/v_00000_2
2,v_00000_3,How do I lower or raise the window?,Use the window control button on the car door.,The door on your left has a press or rocker op...,You can ask the driver for help.,NaN,tool use,train/v_00000_3
3,v_00000_4,Which direction will I need to move to exit th...,Left.,"Usually you can't open the door on the left, y...","You're closer to the left door, but it will be...",NaN,local navigation and positioning,train/v_00000_4
4,v_00000_5,How do I adjust the seatbelt?,You need to pull the belt from the side of you...,It's on the buttom left or right of where you ...,"The seat belt may be on your upper left, you c...",NaN,tool use,train/v_00000_5
...,...,...,...,...,...,...,...,...
2732,v_00920_6,Is the road ahead clear?,"No, there is an electric bike parked in front ...","No, there is a car parked in front of you on t...",NaN,NaN,safety warnings,train/v_00920_6
2733,v_00920_7,Is there a breakfast shop nearby?,"Yes, there is a shop selling tofu pudding and ...","Yes, there are shops with soy milk, tofu puddi...",NaN,NaN,resource acquisition and others,train/v_00920_7
2734,v_00921_5,Where is the place that sells hot dry noodles?,In front left.,On your left front.,NaN,NaN,local navigation and positioning,train/v_00921_5
2735,v_00921_6,Is this freshly fried?,Yes.,"Yes, the boss lady is exploding.",NaN,NaN,information reading,train/v_00921_6


In [4]:
df_val.describe()

,question_id,question,answer0,answer1,answer2,answer3,type,frame_folder_path
count,641,641,638,417,294,63,641,641
unique,641,615,290,300,213,59,7,641
top,v_01245_7,Is it safe to cross the road now?,Yes.,Yes.,I don't know.,Enter and turn right.,information reading,val/v_01245_7
freq,1,5,166,43,27,2,222,1


In [5]:
df_test.describe()

,answer3
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [6]:
def filter_dataset(df):
    """
    This method will take in CSV file as input and apply the appropriate filtering of keywords
    """
    filtered_df = df[df["type"] == "safety warnings"]

    #only filter by potential yes no questions
    include_filter_starters = ("is", "are", "does", "do", "am", "should")
    include_filter_keywords = "obstacle|path|way|clear|safe|hit|walk|street|car|vehicle|traffic|pole|bicycle|truck|bus|danger|road|crosswalk|cross|sidewalk"

    # keywords that are excluded (regex form) str.contain() method accepts regex form of argument
    exclude_filter = "kitchen|sharp|stove|boiling|pot|courier|bowl|fork|microwave|fridge|oven|pantry|spoon|hot|baking|bake|knife|feeding|feed|animal|alive|store"

    # filter rows by questions that could potentially produce a yes/no answer
    filtered_df = filtered_df[filtered_df["question"].str.lower().str.startswith(include_filter_starters) & filtered_df[
        "question"].str.lower().str.contains(include_filter_keywords)]

    filtered_df = filtered_df[~filtered_df["question"].str.lower().str.contains(exclude_filter)]

    #remove answer3 column as they are 90+% Null values (maybe dont drop idk quite yet)
    filtered_df = filtered_df.drop(columns=["answer3"])

    return filtered_df

df_train = filter_dataset(df_train)
df_val = filter_dataset(df_val)
df_test = filter_dataset(df_test)

In [7]:
df_train.describe()

,question_id,question,answer0,answer1,answer2,type,frame_folder_path
count,359,359,359,203,120,359,359
unique,359,279,131,124,90,1,359
top,v_00921_7,Is there an obstacle ahead?,Yes.,Yes.,Yes.,safety warnings,train/v_00921_7
freq,1,48,106,37,13,359,1


In [8]:
df_val.describe()
df_val

,question_id,question,answer0,answer1,answer2,type,frame_folder_path
9,v_00925_4,Are there any obstacles in the aisle in front ...,Yes.,"Yes, there is a staff member in front of you.",NaN,safety warnings,val/v_00925_4
22,v_00931_4,Is it safe to walk further in this direction?,"No, there is a lake in front of you.","It's not safe, there's a river ahead, you'll f...",No.,safety warnings,val/v_00931_4
23,v_00931_5,Are there any obstacles or people nearby that ...,"Yes,there are two persons in front of you.",There are kind people and a couple around you ...,"Yes, there're two people in front of you.",safety warnings,val/v_00931_5
43,v_00941_2,Is there any obstacle in front of me while wal...,No.,NaN,NaN,safety warnings,val/v_00941_2
46,v_00942_1,Are there any obstacles in this hallway?,No.,"No, only pedestrians.",NaN,safety warnings,val/v_00942_1
...,...,...,...,...,...,...,...
617,v_01235_12,Are there any obstacles ahead of me as I walk?,No.,NaN,A staff member.,safety warnings,val/v_01235_12
618,v_01235_13,Is the person in front of me holding anything ...,No.,NaN,NaN,safety warnings,val/v_01235_13
626,v_01237_1,Are there any obstacles in front of me when ri...,No.,NaN,A staff member.,safety warnings,val/v_01237_1
628,v_01239_2,Are there any dangerous or suspicious people n...,No.,NaN,NaN,safety warnings,val/v_01239_2


In [9]:
df_test.describe()
df_test

,question_id,question,answer0,answer1,answer2,type,frame_folder_path
7,v_01249_3,Is the sidewalk in front accessible?,"No, there are people sweeping and a trash can ...",No.,There is a trash can in the front right.,safety warnings,val/v_01249_3
11,v_01251_3,Is there enough room on the sidewalk for me to...,"No, a street lamp is in front of you and there...",No.,Directly ahead is a pillar.,safety warnings,val/v_01251_3
14,v_01252_1,Is there anything blocking my way when I enter...,"Yes, there is a step and packages beside the d...",Yes.,There is a basket in front of the right.,safety warnings,val/v_01252_1
22,v_01256_1,Are there any immediate dangers near me?,"No, it appears safe around you.",No.,NaN,safety warnings,val/v_01256_1
23,v_01256_3,Is there anything blocking the walkway ahead?,"Yes, there is an animal in front of you.",Balustrade.,Short horse.,safety warnings,val/v_01256_3
32,v_01260_4,Is the ground ahead of me clear and safe to ap...,No.,NaN,NaN,safety warnings,val/v_01260_4
33,v_01261_1,Are there any obstacles or uneven ground ahead?,"No, the ground appears clear and even.",No.,There are crowds ahead.,safety warnings,val/v_01261_1
45,v_01267_2,Is it clear to walk past the bicycle?,No.,NaN,NaN,safety warnings,val/v_01267_2
46,v_01268_2,Are there people around me who might be a safe...,No.,NaN,NaN,safety warnings,val/v_01268_2
55,v_01275_1,Is it safe to put my hand into a claw machine ...,No.,Unsafe.,NaN,safety warnings,val/v_01275_1


In [10]:
df_train["val_0"] = 0.0
df_train["val_1"] = 0.0
df_train["val_2"] = 0.0

df_val["val_0"] = 0.0
df_val["val_1"] = 0.0
df_val["val_2"] = 0.0

df_test["val_0"] = 0.0
df_test["val_1"] = 0.0
df_test["val_2"] = 0.0

In [11]:
df_train.describe()


,val_0,val_1,val_2
count,359.0,359.0,359.0
mean,0.0,0.0,0.0
std,0.0,0.0,0.0
min,0.0,0.0,0.0
25%,0.0,0.0,0.0
50%,0.0,0.0,0.0
75%,0.0,0.0,0.0
max,0.0,0.0,0.0


In [12]:
def get_encoded_value(question, answer):

    q = str(question).lower()
    a = str(answer).lower()

    if pd.isna(answer):
        return 0.5 #maybe return 1 instead of 0.5, to further prioritize recall

    if "maybe" in a or "not sure" in a or "don't know" in a or "can't tell" in a:
        return 0.5 #also maybe return 1

    saftey_keywords = ["safe", "clear", "walkable", "empty", "ok to"]
    is_saftey_question = any(word in q for word in saftey_keywords)

    #check if these keywords appear
    if is_saftey_question:
        if "no" in a or "not" in a:
            return 1.0 #danger
        elif "yes":
            return 0.0 #no danger
    #otherwise flip the answers
    else:
        if "no" in a:
            return 0.0 #no danger
        elif "yes":
            return 1.0 #danger

    #mark column as "validate" if no clear answer -> must manually validate and assign encoded value ourselves
    return ".123"

def encode_set(df):
    for index, row in df.iterrows():
        #Get encoded value for each of the three human answers
        v0 = get_encoded_value(row['question'], row['answer0'])
        v1 = get_encoded_value(row['question'], row['answer1'])
        v2 = get_encoded_value(row['question'], row['answer2'])

        df.at[index, "val_0"] = v0
        df.at[index, "val_1"] = v1
        df.at[index, "val_2"] = v2

encode_set(df_train)
encode_set(df_val)
encode_set(df_test)
        
        

In [13]:
df_train

,question_id,question,answer0,answer1,answer2,type,frame_folder_path,val_0,val_1,val_2
1,v_00000_2,Is everything secure inside the car?,Yes.,NaN,NaN,safety warnings,train/v_00000_2,1.0,0.5,0.5
12,v_00002_3,Is it safe to cross the street now?,Yes.,"Yes, but there may be vehicles coming and goin...",NaN,safety warnings,train/v_00002_3,0.0,0.0,0.5
17,v_00003_3,Is it safe to cross the street now?,No.,"No, there are cars on both sides.","A car is about to pass through the street, it ...",safety warnings,train/v_00003_3,1.0,1.0,0.0
36,v_00011_2,Is it safe for me to walk through the exit door?,Yes.,"It should be safe, pay attention to whether th...",NaN,safety warnings,train/v_00011_2,0.0,0.0,0.5
63,v_00014_12,Are there any obstacles on the ground I need t...,Yes.,"Yes, there are several motorcycles parked.",There is an electric car parked in the left fr...,safety warnings,train/v_00014_12,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...
2702,v_00917_2,Is there anything to pay attention to when wal...,"Your right side is full of electric vehicles, ...","There are steps in front of you on the left, a...","There are occasional steps on the left, electr...",safety warnings,train/v_00917_2,1.0,1.0,1.0
2721,v_00918_5,Are there any obstacles when going straight?,No.,There is a small car parked in front of you on...,NaN,safety warnings,train/v_00918_5,0.0,1.0,0.5
2731,v_00920_5,Are there any obstacles ahead?,"Yes, there is a car standing in front of you o...",There is a car parked in front of you on the r...,NaN,safety warnings,train/v_00920_5,1.0,1.0,0.5
2732,v_00920_6,Is the road ahead clear?,"No, there is an electric bike parked in front ...","No, there is a car parked in front of you on t...",NaN,safety warnings,train/v_00920_6,1.0,1.0,0.5


In [14]:
#convert back to csv file for download
df_train.to_csv("egoblind_for_review", index=False)

In [15]:
#to create the download link for the csv file
from IPython.display import FileLink

FileLink(r'egoblind_for_review')

/kaggle/working/egoblind_for_review

Initialize 3 new colums for each set to 0.0. These will be the encoded values based off the answer columns. Val_k corresponds to Answer_k